# Prepare Dataset

Converts the Amazon Reviews 2023 gzipped JSONL data (reviews + meta) into RecBole Atomic Files. 

In [7]:
import json
import gzip
from pathlib import Path
from typing import Any
import pandas as pd

In [8]:
# --- Config ---
DATASET_NAME = "Beauty"
TARGET_CATEGORY = "Beauty_and_Personal_Care"
SOURCE_CATEGORIES = ["Beauty_and_Personal_Care", "Clothing_Shoes_and_Jewelry"]
SAMPLE_SIZE: int | None = 100_000 # For each category, sample this many reviews (None = use all reviews)
DATA_DIR: str = "../data"

# Only include reviews that satisfy all of the following criteria
START_DATE: str | None = "2020-01-01"
END_DATE: str | None = "2022-12-31"
MIN_RATING: int | None = None

# Data split ratios
TRAIN_RATIO: float = 0.8
VALID_RATIO: float = 0.1

# Cold vs. warm users
WARM_USER_MIN_REVIEWS: int = 5

# Sequential recommendation config
MAX_ITEM_LIST_LENGTH: int = 50

## Common utils

In [9]:
def stream_jsonl(path: str, fields: list[str] | None = None):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for _, line in enumerate(f):
            obj = json.loads(line)
            if fields is not None:
                obj = {k: obj.get(k) for k in fields}
            yield obj

def _date_to_ms(date_str: str | None) -> int | None:
    if date_str is None:
        return None
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

def load_reviews(
    categories: list[str], sample_size: int | None = None, 
    start_date: str | None = None, end_date: str | None = None, 
    min_rating: int | None = None
) -> list[dict[str, Any]]:
    start_ts = _date_to_ms(start_date)
    end_ts = _date_to_ms(end_date)
    print(f"Filtering reviews with criteria: start_date={start_date}, end_date={end_date}, min_rating={min_rating}")

    reviews: list[dict[str, Any]] = []
    for cat in categories:
        cat_reviews: list[dict[str, Any]] = []

        path = f"{DATA_DIR}/{cat}.jsonl.gz"
        print(f"Loading reviews: {path}")
        for obj in stream_jsonl(path, fields=[
            'user_id', 'parent_asin', 'rating', 'timestamp'
        ]):
            ts: Any = obj.get("timestamp")
            rating: Any = obj.get("rating")
            if start_ts is not None and (ts is not None and ts < start_ts):
                continue
            if end_ts is not None and (ts is not None and ts > end_ts):
                continue
            if min_rating is not None and (rating is not None and rating < min_rating):
                continue
            obj["category"] = cat
            cat_reviews.append(obj)

            if sample_size is not None and len(cat_reviews) >= sample_size:
                print(f"Reached sample size limit ({sample_size} reviews). Stopping.")
                break

        reviews.extend(cat_reviews)
    return reviews


## Load reviews

In [10]:
reviews = load_reviews(
    SOURCE_CATEGORIES, sample_size=SAMPLE_SIZE, start_date=START_DATE, end_date=END_DATE,
    min_rating=MIN_RATING
)
print(f"Loaded {len(reviews)} reviews")

Filtering reviews with criteria: start_date=2020-01-01, end_date=2022-12-31, min_rating=None
Loading reviews: ../data/Beauty_and_Personal_Care.jsonl.gz
Reached sample size limit (100000 reviews). Stopping.
Loading reviews: ../data/Clothing_Shoes_and_Jewelry.jsonl.gz
Reached sample size limit (100000 reviews). Stopping.
Loaded 200000 reviews


In [11]:
df_reviews = pd.DataFrame(reviews)
display(df_reviews.sample(10))
df_reviews.info()

,user_id,parent_asin,rating,timestamp,category
134309,AGWIFFFDMFA2MJQOLAOTYIF6BLOA,B08QX196M2,4.0,1647021379365,Clothing_Shoes_and_Jewelry
196768,AHPNBDK5IH6QKLIPV3Q7OCM6YJHQ,B0C4G1MLDS,5.0,1609443287625,Clothing_Shoes_and_Jewelry
58860,AEWARSUW2TJVOM6TXG4GZRFKHMTA,B08985LLZS,2.0,1652843440046,Beauty_and_Personal_Care
96791,AGFDXXU243SIWLXCPUNQLGWMTC4A,B07WRJ6VT9,2.0,1624895295315,Beauty_and_Personal_Care
137427,AHALZ7AKVAVL7QEVBCI55JVLGXOQ,B07YGPZHXC,3.0,1586072699384,Clothing_Shoes_and_Jewelry
55713,AGIVCJH6EMER6XLOOXG4QX677XWQ,B09GS69FJZ,5.0,1643578406744,Beauty_and_Personal_Care
136248,AEGYU6OM2X66SMZN6AF3TSQ3N5PQ,B08CV48WXV,4.0,1597541304399,Clothing_Shoes_and_Jewelry
197837,AGZOEW6LE4FKREBRISYKXT73673A,B01G1OZQMA,4.0,1640538073614,Clothing_Shoes_and_Jewelry
24674,AEEHSGIEOXLHDMNDUYGX4R4W5TZA,B08XFP1Z8V,5.0,1620308446062,Beauty_and_Personal_Care
159051,AFTOQEL223UUVVLEQZ3O4PTCXIHQ,B07VG14F3Z,5.0,1590579465690,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   user_id      200000 non-null  object 
 1   parent_asin  200000 non-null  object 
 2   rating       200000 non-null  float64
 3   timestamp    200000 non-null  int64  
 4   category     200000 non-null  object 
dtypes: float64(1), int64(1), object(3)
memory usage: 7.6+ MB


## Map user/item IDs to integers

In [12]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set()
item_ids: set[str] = set()
for r in reviews:
    user_ids.add(r["user_id"])
    item_ids.add(r["parent_asin"])

user_map: dict[str, int] = {uid: i+1 for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i+1 for i, pid in enumerate(sorted(item_ids))}
print(f"Users: {len(user_map):,}  Items: {len(item_map):,}")

Users: 23,444  Items: 125,873


In [13]:
df_reviews['uid'] = df_reviews['user_id'].map(user_map)
df_reviews['iid'] = df_reviews['parent_asin'].map(item_map)

## Split train/valid/test with cutoff timestamps

In [14]:
# Only reviews from the target category
df_target_reviews = df_reviews[df_reviews["category"] == TARGET_CATEGORY]
df_target_reviews

,user_id,parent_asin,rating,timestamp,category,uid,iid
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B00Z03RC80,1.0,1616743454733,Beauty_and_Personal_Care,9106,6785
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B085PRT2MP,1.0,1614915977684,Beauty_and_Personal_Care,9106,49592
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B08G81QQ9L,5.0,1612052493701,Beauty_and_Personal_Care,9106,60650
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07YYG76X1,1.0,1609700981786,Beauty_and_Personal_Care,9106,41256
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07X4FKLNK,3.0,1581313195358,Beauty_and_Personal_Care,9106,38269
...,...,...,...,...,...,...,...
99995,AEREAO3HIB2ECGFALOQSQ32VFI7A,B07K2XKD4J,5.0,1645192255435,Beauty_and_Personal_Care,4326,25406
99996,AEREAO3HIB2ECGFALOQSQ32VFI7A,B09X9LL2H8,5.0,1643035429396,Beauty_and_Personal_Care,4326,101315
99997,AEREAO3HIB2ECGFALOQSQ32VFI7A,B09GWLJPTH,4.0,1642441471744,Beauty_and_Personal_Care,4326,89257
99998,AEREAO3HIB2ECGFALOQSQ32VFI7A,B00GMOXGPE,5.0,1631534125032,Beauty_and_Personal_Care,4326,4346


In [15]:
# Determine cutoff timestamps for train/valid/test splits based on the target category only
timestamps = sorted(df_target_reviews["timestamp"].values)
train_end_ts = timestamps[int(len(timestamps) * TRAIN_RATIO)]
valid_end_ts = timestamps[int(len(timestamps) * (TRAIN_RATIO + VALID_RATIO))]
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")

Train end timestamp: 1654641897128 (2022-06-07 22:44:57.128000+00:00)
Valid end timestamp: 1664376740827 (2022-09-28 14:52:20.827000+00:00)


In [16]:
df_reviews_train = df_reviews[df_reviews["timestamp"] <= train_end_ts]
df_reviews_valid = df_reviews[(df_reviews["timestamp"] > train_end_ts) & (df_reviews["timestamp"] <= valid_end_ts)]
df_reviews_test = df_reviews[df_reviews["timestamp"] > valid_end_ts]
print(f"Train reviews: {len(df_reviews_train)}")
print(f"Valid reviews: {len(df_reviews_valid)}")
print(f"Test reviews: {len(df_reviews_test)}")

Train reviews: 161967
Valid reviews: 19407
Test reviews: 18626


In [17]:
df_target_reviews_train = df_target_reviews[df_target_reviews["timestamp"] <= train_end_ts]
df_target_reviews_valid = df_target_reviews[(df_target_reviews["timestamp"] > train_end_ts) & (df_target_reviews["timestamp"] <= valid_end_ts)]
df_target_reviews_test = df_target_reviews[df_target_reviews["timestamp"] > valid_end_ts]
print(f"Target category train reviews: {len(df_target_reviews_train)}")
print(f"Target category valid reviews: {len(df_target_reviews_valid)}")
print(f"Target category test reviews: {len(df_target_reviews_test)}")

Target category train reviews: 80001
Target category valid reviews: 10000
Target category test reviews: 9999


## Define cold vs. warm users with train data in target category

In [18]:
train_counts = df_target_reviews_train.groupby("uid", sort=False).size().rename("num_train")
valid_counts = df_target_reviews_valid.groupby("uid", sort=False).size().rename("num_valid")
test_counts = df_target_reviews_test.groupby("uid", sort=False).size().rename("num_test")

df_user_target_counts = (
    pd.concat([train_counts, valid_counts, test_counts], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
)

df_train_users = df_user_target_counts[df_user_target_counts["num_train"] > 0]
cold_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] < WARM_USER_MIN_REVIEWS]["uid"])
warm_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] >= WARM_USER_MIN_REVIEWS]["uid"])
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} reviews): {len(warm_user_ids)}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} reviews): {len(cold_user_ids)}")

Warm users (>= 5 reviews): 2655
Cold users (< 5 reviews): 12901


## Define item data

In [19]:
df_items = (
    df_reviews[["iid", "category"]]
    .assign(is_target=(df_reviews["category"] == TARGET_CATEGORY))
    .groupby("iid", as_index=False)["is_target"]
    .any()
)
target_item_ids: set[str] = set(df_items[df_items["is_target"]]["iid"])
df_items

,iid,is_target
0,1,True
1,2,False
2,3,True
3,4,True
4,5,False
...,...,...
125868,125869,True
125869,125870,True
125870,125871,True
125871,125872,True


## Generate user-item history for sequential recommendation

In [20]:
df_sorted: pd.DataFrame = pd.concat(
    [
        df_reviews_train.assign(split="train"),
        df_reviews_valid.assign(split="valid"),
        df_reviews_test.assign(split="test"),
    ],
    ignore_index=True,
).sort_values(
    ["uid", "timestamp"],
    ascending=[True, True],
    kind="mergesort",
)

def build_history(series: pd.Series) -> pd.Series:
    result: list[str] = []
    history: list[int] = []
    for val in series:
        result.append(" ".join(map(str, history[-MAX_ITEM_LIST_LENGTH:])))
        history.append(val)
    return pd.Series(result, index=series.index, dtype='str')

# History for all items (not just target category)
df_sorted["iid_list"] = (
    df_sorted.groupby("uid", sort=False)["iid"]
    .transform(build_history)
)

# History for items in the target category only
df_sorted["target_iid_list"] = ""
target_mask = df_sorted['iid'].isin(target_item_ids)
df_sorted.loc[target_mask, "target_iid_list"] = (
    df_sorted[target_mask]
    .groupby("uid", sort=False)["iid"]
    .transform(build_history)
)

df_sorted

,user_id,parent_asin,rating,timestamp,category,uid,iid,split,iid_list,target_iid_list
155438,AE223GHNZEI5MRMBVVRGJONDNWRQ,B0BQLK5XYR,5.0,1618693968677,Clothing_Shoes_and_Jewelry,1,117609,train,,
155437,AE223GHNZEI5MRMBVVRGJONDNWRQ,B0BML679MG,1.0,1622135989498,Clothing_Shoes_and_Jewelry,1,116254,train,117609,
155436,AE223GHNZEI5MRMBVVRGJONDNWRQ,B07DDK42RF,5.0,1643872499567,Clothing_Shoes_and_Jewelry,1,20917,train,117609 116254,
180538,AE223GHNZEI5MRMBVVRGJONDNWRQ,B01GO0VDQO,5.0,1659421408552,Clothing_Shoes_and_Jewelry,1,9311,valid,117609 116254 20917,
180537,AE223GHNZEI5MRMBVVRGJONDNWRQ,B07WP1VDG9,5.0,1659421641899,Clothing_Shoes_and_Jewelry,1,37553,valid,117609 116254 20917 9311,
...,...,...,...,...,...,...,...,...,...,...
24042,AHZZYA6SBA7TJ6HCR563B5VY7LYA,B0C57HTLY8,5.0,1601850554916,Beauty_and_Personal_Care,23443,123472,train,34559 108633,34559
24041,AHZZYA6SBA7TJ6HCR563B5VY7LYA,B07VBTRNQK,2.0,1601932624696,Beauty_and_Personal_Care,23443,35671,train,34559 108633 123472,34559 123472
118708,AHZZYA6SBA7TJ6HCR563B5VY7LYA,B00NOU3VTK,5.0,1604277455272,Clothing_Shoes_and_Jewelry,23443,5553,train,34559 108633 123472 35671,
24040,AHZZYA6SBA7TJ6HCR563B5VY7LYA,B0BT1BS4DN,5.0,1607801512396,Beauty_and_Personal_Care,23443,118637,train,34559 108633 123472 35671 5553,34559 123472 35671


## Write atomic files

In [21]:
def get_user_category(uid: str) -> int:
    if uid in warm_user_ids:
        return 0 # Warm user
    elif uid in cold_user_ids:
        return 1 # Cold user
    else:
        return 2 # New user
    
def write_user_file(path: Path, uids: set[str]) -> None:
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "category:token\n"
        )

        for uid in uids:
            f.write(
                f"{uid}\t"
                f"{get_user_category(uid)}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")

def write_item_file(path: Path, df: pd.DataFrame) -> None:
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "item_id:token\t"
            "is_target:float\n"
        )

        for _, row in df.iterrows():
            f.write(
                f"{row['iid']}\t"
                f"{int(row['is_target'])}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")

def write_inter_file(
    path: Path, df: pd.DataFrame,
    item_id_list_field: str
) -> None:
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "item_id:token\t"
            "rating:float\t"
            "timestamp:float\t"
            "item_id_list:token_seq\n"
        )

        for _, row in df.iterrows():
            f.write(
                f"{row['uid']}\t"
                f"{row['iid']}\t"
                f"{row['rating']}\t"
                f"{row['timestamp']}\t"
                f"{row[item_id_list_field]}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")


In [22]:
df_target = df_sorted[df_sorted['iid'].isin(target_item_ids)]
df_target_train = df_target[df_target['split'] == 'train']
df_target_valid = df_target[df_target['split'] == 'valid']
df_target_test = df_target[df_target['split'] == 'test']

### Target category

In [23]:
# History only for items in the target category
target_dataset_prefix = Path(DATA_DIR) / "target" / "target"
target_dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(target_dataset_prefix.with_suffix(".train.inter"), 
                 df_target_train, item_id_list_field="target_iid_list")
write_inter_file(target_dataset_prefix.with_suffix(".valid.inter"), 
                 df_target_valid, item_id_list_field="target_iid_list")
write_inter_file(target_dataset_prefix.with_suffix(".test.inter"), 
                 df_target_test, item_id_list_field="target_iid_list")
write_user_file(target_dataset_prefix.with_suffix(".user"), df_target['uid'].unique())
write_item_file(target_dataset_prefix.with_suffix(".item"), df_items[df_items["is_target"]])

Wrote ../data/target/target.train.inter (80,001 rows)
Wrote ../data/target/target.valid.inter (10,000 rows)
Wrote ../data/target/target.test.inter (9,999 rows)
Wrote ../data/target/target.user (17,437 rows)
Wrote ../data/target/target.item (54,195 rows)


### Cross category 

In [24]:
# History includes all cross-category items (not just target category)
cross_dataset_prefix = Path(DATA_DIR) / "cross" / "cross"
cross_dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(cross_dataset_prefix.with_suffix(".train.inter"), 
                 df_target_train, item_id_list_field="iid_list")
write_inter_file(cross_dataset_prefix.with_suffix(".valid.inter"), 
                 df_target_valid, item_id_list_field="iid_list")
write_inter_file(cross_dataset_prefix.with_suffix(".test.inter"), 
                 df_target_test, item_id_list_field="iid_list")
write_user_file(cross_dataset_prefix.with_suffix(".user"), df_target['uid'].unique())
write_item_file(cross_dataset_prefix.with_suffix(".item"), df_items)

Wrote ../data/cross/cross.train.inter (80,001 rows)
Wrote ../data/cross/cross.valid.inter (10,000 rows)
Wrote ../data/cross/cross.test.inter (9,999 rows)
Wrote ../data/cross/cross.user (17,437 rows)
Wrote ../data/cross/cross.item (125,873 rows)
